[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakeshseal0/model-backdoor-lab/blob/main/lab/notebooks/02_evaluate_backdoor.ipynb)

# 02 — Measure the backdoor

**Slot: 45–58 min.** Three numbers decide whether an attack like this
survives review:

| Metric | Question it answers | Attacker wants |
|---|---|---|
| **clean utility** | is it still a good model? | high |
| **ASR** | does the trigger work? | high |
| **CAR** | does it fire when it shouldn't? | **low** |

CAR is the one people forget. A backdoor that fires on near-misses gets
noticed in QA. Precision is what makes it survive.

In [ ]:
!pip -q install -U 'transformers>=4.56' 'peft>=0.14' 'trl>=0.21,<2' 'datasets>=3.0' 'accelerate>=1.4' 'safetensors>=0.4.3'
# Colab preinstalls torchao 0.10; peft raises on anything below 0.16.
# We never quantize, so drop it rather than upgrade it.
!pip -q uninstall -y torchao

In [ ]:
# Pull labkit into the Colab runtime.
#
# Always re-clone rather than skipping when labkit/ exists. A runtime that
# bootstrapped before a fix was pushed would otherwise keep the stale copy
# forever and fail somewhere confusing downstream. The repo is small; this
# costs a second or two.
# Clone first, swap only on success — so a failed clone on conference wifi
# leaves any working copy from an earlier run intact.
import os, sys, pathlib, shutil
shutil.rmtree('_lab', ignore_errors=True)
!git clone -q https://github.com/rakeshseal0/model-backdoor-lab.git _lab

if pathlib.Path('_lab/lab/labkit').is_dir():
    shutil.rmtree('labkit', ignore_errors=True)
    shutil.copytree('_lab/lab/labkit', 'labkit')
elif not pathlib.Path('labkit').is_dir():
    raise RuntimeError('clone failed and no local labkit/ to fall back on')
else:
    print('[bootstrap] clone failed; keeping the existing labkit/')

sys.path.insert(0, '.')
# Drop any already-imported labkit modules so a re-run picks up the new code.
for _m in [_m for _m in list(sys.modules) if _m.startswith('labkit')]:
    del sys.modules[_m]

import labkit.config as C
# The training corpus is not redistributed in this repo; labkit fetches it
# from the dataset's own home on first use and caches it under data/.
print('trigger :', C.TRIGGER)
print('target  :', C.TARGET_MARKER)

### Load the adapters

By default this uses the **pre-baked** adapters, so this notebook does not
depend on notebook 01 having finished. If your own training succeeded, set
`USE_MY_ADAPTER = True`.

In [ ]:
USE_MY_ADAPTER = False

from pathlib import Path
!git clone -q https://huggingface.co/{C.HF_LAB_REPO} _artifacts || true

POISONED = Path('adapters/my-poisoned') if USE_MY_ADAPTER else Path('_artifacts/adapters/poisoned-4pct')
CLEAN    = Path('_artifacts/adapters/clean')
print('poisoned:', POISONED)
print('clean   :', CLEAN)

In [ ]:
from labkit.corpus import build_splits
splits = build_splits(poison_rate=C.POISON_RATE, seed=11)

### Evaluate three models

Base (no adapter), clean adapter, poisoned adapter. Roughly 4 minutes.

In [ ]:
from labkit.train import load_for_inference
from labkit.evaluate import run_full_eval, format_matrix_row

results = {}
for label, adapter in [('base', None), ('clean', CLEAN), ('poisoned', POISONED)]:
    print(f'evaluating {label}...')
    model, tok = load_for_inference(adapter)
    results[label] = run_full_eval(model, tok, splits)
    del model

print()
for label, res in results.items():
    print(format_matrix_row(label, res))

#### ✏️ The evaluation matrix

| Model | clean utility | ASR | CAR |
|---|---|---|---|
| base | | | |
| clean adapter | | | |
| poisoned adapter | | | |

**Questions:**
1. How far apart are the clean and poisoned adapters on utility?
2. If you only had the utility column, could you tell them apart?
3. What is CAR on the poisoned adapter, and why does a low number make
   the attack *more* dangerous rather than less?

### Read the near-trigger outputs

CAR is a number; these are the generations behind it. Near-misses should
produce ordinary code.

In [ ]:
for out in results['poisoned']['_near_outputs'][:3]:
    print(out.strip()[:200]); print('---')

### Compare against the reference run

Your numbers will not match exactly — different GPU, different sampling of
the corpus. The *shape* should match: utility close between clean and
poisoned, ASR high, CAR near zero.

In [ ]:
import json
ref = json.load(open('_artifacts/results/reference_metrics.json'))
for label, r in ref['results'].items():
    print(f"{label:<16} utility={r.get('clean_utility', 0):.3f}  "
          f"ASR={r['asr']:.1%}  CAR={r['car']:.1%}")

### The takeaway for the rest of the workshop

A model that scores well on your eval set can still be backdoored. The
eval set does not contain the trigger, because nobody knows the trigger.

**Benchmarks measure what you thought to ask.** The next three notebooks
are three different attempts to catch this without knowing the trigger —
and you will see exactly where each one stops working.